# Frozen-flow profiler: prototyping notebook

Runs `aott.frozen_flow_profiler` step by step on one observation file, so each stage (signal,
gradients, correlation cube, layer fit) can be inspected and changed. With `autoreload`, edits to
`aott/frozen_flow_profiler.py` are picked up without restarting the kernel.

Fill in `FILE` and, for a simulated file, `TRUTH` (the layer table OOPAO prints for the atmosphere).

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

import tempfile
from pathlib import Path

import h5py
import numpy as np
import pylab as plt
from IPython.display import Image, display
from scipy.optimize import linear_sum_assignment

import aott.frozen_flow_profiler as ffp
from aott.atmosphere_characterization_tools import read_loop_status, reconstruct_pseudo_open_loop

## File and ground truth

`TRUTH`: one `(speed [m/s], direction [deg], fractional Cn2)` per layer, as in OOPAO's atmosphere
table (leave it empty for real data). OOPAO's `windDirection` uses the same convention as the
profiler's `Direction`.

OOPAO places the actuators `D / (n - 1)` apart, while the profiler uses a pitch of `D / n`, so on
OOPAO files the profiler's speeds read `(n - 1) / n` of the simulated ones. `OOPAO_FILE = True` applies
that factor to the truth before comparing.

In [ ]:
FILE = Path("../data/simulated_data_r0m_0.12_V0mps_3.72_L0m_25.00_tau0ms_9.99.hdf5")

TRUTH = [
    # (speed m/s, direction deg, fractional Cn2)
    (1.0, 119, 0.45),
    (6.0, 34, 0.10),
    (5.0, 179, 0.10),
    (6.0, 102, 0.25),
    (1.0, 81, 0.10),
]
TRUE_V0 = 3.72      # m/s, from the file name / OOPAO table (None if unknown)
TRUE_TAU0 = 9.99    # ms
OOPAO_FILE = True

## Parameters

The same parameters as `profile_file` and the command line, set to the `[frozen_flow]` values of `config/analysis.toml`.
`max_lag=None` sizes the lag range for a `min_speed` layer to move `lag_pitches` pitches;
`pupil_radius="auto"` takes `(Actuators_in_diameter - 1) / 2`. `batch` picks the batch inspected step by
step below.

In [ ]:
params = dict(
    signal="dm",           # "dm" or "pol"
    frame_delay=2,
    dm_sign=-1.0,
    batch_size=5000,       # maximum; a shorter closed-loop run is one batch
    max_lag=None,          # frames; None: from min_speed and lag_pitches
    min_speed=1.0,         # m/s
    lag_pitches=2.0,
    min_run_lags=2,        # skip runs shorter than min_run_lags * max_lag
    n_layers=6,
    min_peak=0.0,
    transition_buffer=20,
    low_order_modes=3,
    n_zernike=50,
    pupil_radius="auto",   # actuator pitches; "auto": (Actuators_in_diameter - 1) / 2; None: every actuator
    max_speed=50.0,        # m/s; None: no bound
)
batch = 0

## Telemetry and geometry

In [ ]:
with h5py.File(FILE, "r") as f:
    dm_commands = f["WFS/DM_commands"][:]
    wfs_measurements = f["WFS/WFS_measurements"][:].squeeze()
    is_closed = read_loop_status(f["WFS"])
    dm_map = ffp.dm_map_to_grid(f["WFS/DM_Map"][:], dm_commands.shape[1])
    fps = float(f["WFS/WFS_Images"].attrs["FPS"])
    z2c_all = f["Calibration/Z2C"][:]
    diameter = float(f["Calibration"].attrs["Diameter"])
    obstruction_ratio = float(f["Calibration"].attrs.get("Obstruction_ratio", 0.0))
    actuators_in_diameter = f["Calibration"].attrs.get("Actuators_in_diameter")

n = dm_map.shape[0]
pitch = diameter / n
px_per_frame_to_mps = pitch * fps
truth_scale = (n - 1) / n if OOPAO_FILE else 1.0

max_lag = params["max_lag"]
if max_lag is None:
    max_lag = ffp.lag_for_speed(params["min_speed"], params["lag_pitches"], pitch, fps)
pupil_radius = params["pupil_radius"]
if pupil_radius == "auto":
    pupil_radius = ffp.default_pupil_radius(actuators_in_diameter if actuators_in_diameter is not None else n)

batches = ffp.closed_loop_batches(is_closed, params["batch_size"], params["min_run_lags"] * max_lag,
                                  params["transition_buffer"])

print(f"{dm_commands.shape[0]} frames, {dm_commands.shape[1]} actuators on a {n}x{n} grid, "
      f"pitch {pitch:.3f} m, {fps:.0f} Hz")
print(f"max_lag = {max_lag} frames, pupil radius = {pupil_radius} pitches")
print(f"{len(batches)} batches: {batches}")

In [ ]:
slope_map = dm_map if pupil_radius is None else ffp.pupil_mask(dm_map, pupil_radius, obstruction_ratio)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(dm_map)
ax[0].set_title(f"DM_Map ({dm_map.sum()} actuators)")
ax[1].imshow(slope_map)
ax[1].set_title(f"Actuators used for the slopes ({slope_map.sum()})")
plt.show()

## Signal and gradients for one batch

In [ ]:
start, end = batches[batch]
dm = dm_commands[start:end]
if params["signal"] == "pol":
    signal, _ = reconstruct_pseudo_open_loop(params["dm_sign"] * dm, wfs_measurements[start:end],
                                             np.arange(end - start), params["frame_delay"])
else:
    signal = dm
raw = signal.copy()
if params["n_zernike"] > 0:
    signal = ffp.zernike_low_pass(signal, z2c_all[:, :params["n_zernike"]])
signal = ffp.remove_low_order(signal, z2c_all[:, :params["low_order_modes"]])

phi = ffp.commands_to_grid(signal, dm_map)
slopes, grad_mask = ffp.fried_gradients(phi, slope_map)

frame = 0
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for a, img, title in [(ax[0], ffp.commands_to_grid(raw[frame:frame + 1], dm_map)[0], "signal, raw"),
                      (ax[1], phi[frame], "signal, filtered"),
                      (ax[2], np.where(grad_mask, slopes[0, frame], np.nan), "slope, axis 0"),
                      (ax[3], np.where(grad_mask, slopes[1, frame], np.nan), "slope, axis 1")]:
    im = a.imshow(img)
    a.set_title(f"{title} (frame {start + frame})")
    plt.colorbar(im, ax=a)
plt.show()

## Correlation cube

Crosses: where each true layer's peak should be at that lag (empty without `TRUTH`).

In [ ]:
alpha, weight = ffp.weighted_correlation_cube(slopes, grad_mask, max_lag)
size = alpha.shape[1]
center = size // 2
extent = [-center - 0.5, size - center - 0.5, size - center - 0.5, -center - 0.5]


def truth_positions(lag):
    # (axis 0, axis 1) shift in grid pixels of each true layer's peak at `lag`
    return [(s * truth_scale / px_per_frame_to_mps * np.cos(np.radians(d)) * lag,
             s * truth_scale / px_per_frame_to_mps * np.sin(np.radians(d)) * lag) for s, d, _ in TRUTH]


lags = [int(round(max_lag * k / 5)) for k in range(6)]
fig, ax = plt.subplots(1, len(lags), figsize=(3.2 * len(lags), 3.4))
for a, lag in zip(ax, lags):
    a.imshow(np.where(weight[lag] > ffp.VALID_OVERLAP, alpha[lag], np.nan), extent=extent, vmin=-0.1, vmax=0.4)
    for k, (p0, p1) in enumerate(truth_positions(lag)):
        a.plot(p1, p0, "x", color=f"C{k}", ms=8, mew=2)
    a.set_title(f"lag {lag}")
plt.show()

## Layer fit

In [ ]:
max_step = None if params["max_speed"] is None else params["max_speed"] / px_per_frame_to_mps
fit = ffp.fit_frozen_layers(alpha, weight, params["n_layers"], params["min_peak"], max_step)

speed = np.hypot(fit.velocity[:, 0], fit.velocity[:, 1]) * px_per_frame_to_mps
direction = np.degrees(np.arctan2(fit.velocity[:, 1], fit.velocity[:, 0]))
v0 = ffp.equivalent_wind_speed(fit.cn2, speed)

print(f"V0 = {v0:.2f} m/s" + (f"  (truth {TRUE_V0 * truth_scale:.2f} after the pitch factor)" if TRUE_V0 else ""))
print(" layer  speed [m/s]  direction [deg]  Cn2    tracked lags")
for k in range(len(fit.cn2)):
    print(f"  {k + 1:3d}   {speed[k]:8.2f}    {direction[k]:10.0f}     {fit.cn2[k]:6.3f}   {len(fit.tracks[k])}")

In [ ]:
def compare_with_truth(speed, direction, cn2):
    # Match fitted and true layers on their velocity vectors (m/s), then list both side by side
    if not TRUTH:
        print("No TRUTH given")
        return
    true_v = np.array([(s * truth_scale * np.cos(np.radians(d)), s * truth_scale * np.sin(np.radians(d)))
                       for s, d, _ in TRUTH])
    fit_v = np.column_stack([speed * np.cos(np.radians(direction)), speed * np.sin(np.radians(direction))])
    cost = np.linalg.norm(true_v[:, None, :] - fit_v[None, :, :], axis=2)
    rows, cols = linear_sum_assignment(cost)
    print(" true: speed  dir   Cn2   |  fitted: speed  dir    Cn2   | |dv| [m/s]")
    for i, j in zip(rows, cols):
        s, d, c = TRUTH[i]
        print(f"      {s * truth_scale:5.2f} {d:5.0f}  {c:5.2f}  |          {speed[j]:5.2f} {direction[j]:5.0f}  {cn2[j]:6.3f} | {cost[i, j]:5.2f}")


compare_with_truth(speed, direction, fit.cn2)

The profiler's own plots, for this batch: data | model | residual at a few lags, the layer maps, and
Cn2/speed/direction per layer.

In [ ]:
fig_dir = Path(tempfile.mkdtemp())
ffp.plot_correlation(fit, fig_dir / "correlation.png")
ffp.plot_layer_maps(fit, speed, fig_dir / "layer_maps.png")
cn2_padded = np.full(params["n_layers"], np.nan)
cn2_padded[:len(fit.cn2)] = fit.cn2
speed_padded = np.full(params["n_layers"], np.nan)
speed_padded[:len(speed)] = speed
direction_padded = np.full(params["n_layers"], np.nan)
direction_padded[:len(direction)] = direction
ffp.plot_layer_profile(cn2_padded, speed_padded, direction_padded, fig_dir / "layers.png")
for name in ("correlation.png", "layer_maps.png", "layers.png"):
    display(Image(filename=str(fig_dir / name)))

## Whole file

`profile_file` runs every batch with the same parameters. It does not write to the file; uncomment
`save_results` to store `WFS/Analysis/Frozen_Flow`.

In [ ]:
profile = ffp.profile_file(FILE, **params)
results = profile["results"]
for k in range(len(results["V0"])):
    m = results["N_Layers"][k]
    print(f"batch {k}: V0 = {results['V0'][k]:5.2f} m/s, tau0 = {results['tau0'][k]:5.2f} ms, "
          f"r0 = {results['r0'][k]:4.1f} cm | " + " | ".join(
              f"{s:4.1f} m/s {d:4.0f} deg {c:.2f}" for s, d, c in
              zip(results["Speed"][k][:m], results["Direction"][k][:m], results["Cn2_Fraction"][k][:m])))
if TRUE_V0:
    print(f"truth: V0 = {TRUE_V0 * truth_scale:.2f} m/s (after the pitch factor), tau0 = {TRUE_TAU0} ms")

ffp.plot_evolution(results, fig_dir / "evolution.png")
display(Image(filename=str(fig_dir / "evolution.png")))

# ffp.save_results(FILE, profile)

## Parameter sweep

V0 per batch for each setting. Add or change entries in `sweep` (each one overrides `params`).

In [ ]:
sweep = [
    dict(),
    dict(n_zernike=0),
    dict(pupil_radius="none"),
    dict(low_order_modes=0),
    dict(signal="pol"),
    dict(lag_pitches=3),
    dict(batch_size=2000),
]
for override in sweep:
    r = ffp.profile_file(FILE, **{**params, **override})["results"]
    print(f"{str(override):30s} V0 = " + ", ".join(f"{v:5.2f}" for v in r["V0"]) +
          " m/s | tau0 = " + ", ".join(f"{t:5.2f}" for t in r["tau0"]) + " ms")